<a href="https://colab.research.google.com/github/Al-Amin95/PromptInjectionDetectionSystem/blob/data-analysis/notebooks/1-data-analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1 Data Analysis: Prompt Injection Detection Dataset

## Part-1:  Notebook Setup and Reproducibility

In [ ]:
# 1. Mount Google Drive

from google.colab import drive
from pathlib import Path

DRIVE_ROOT = Path("/content/drive")
MY_DRIVE = DRIVE_ROOT / "MyDrive"

if MY_DRIVE.exists():
    print("Google Drive is already mounted.")
else:
    drive.mount("/content/drive")
    print("Google Drive mounted successfully.")

print("MyDrive path:", MY_DRIVE)
print("MyDrive exists:", MY_DRIVE.exists())

In [ ]:
# 2-Auto-detect project folder path

from pathlib import Path

MY_DRIVE = Path("/content/drive/MyDrive")

print("MyDrive exists:", MY_DRIVE.exists())

# 1. Find USW dissertation folder automatically
usw_candidates = []

for item in MY_DRIVE.iterdir():
    if item.is_dir():
        normalized_name = item.name.lower().replace("_", " ").replace("-", " ")
        if "usw" in normalized_name and "dissertation" in normalized_name:
            usw_candidates.append(item)

print("\nUSW candidate folders:")
for idx, folder in enumerate(usw_candidates):
    print(idx, "->", repr(folder.name), "| full path:", folder)

if len(usw_candidates) == 0:
    raise FileNotFoundError("No USW Dissertation folder found inside MyDrive.")

USW_DIR = usw_candidates[0]

# 2. Find project folder automatically
project_candidates = []

for item in USW_DIR.iterdir():
    if item.is_dir():
        normalized_name = item.name.lower().replace("_", " ").replace("-", " ")
        if "prompt" in normalized_name and "injection" in normalized_name:
            project_candidates.append(item)

print("\nProject candidate folders:")
for idx, folder in enumerate(project_candidates):
    print(idx, "->", repr(folder.name), "| full path:", folder)

if len(project_candidates) == 0:
    raise FileNotFoundError("No Prompt Injection project folder found inside the USW folder.")

PROJECT_ROOT = project_candidates[0]

# 3. Find Notebooks folder automatically, case-insensitive
notebook_candidates = []

for item in PROJECT_ROOT.iterdir():
    if item.is_dir() and item.name.lower() == "notebooks":
        notebook_candidates.append(item)

print("\nNotebook candidate folders:")
for idx, folder in enumerate(notebook_candidates):
    print(idx, "->", repr(folder.name), "| full path:", folder)

if len(notebook_candidates) == 0:
    raise FileNotFoundError("No Notebooks folder found inside the project folder.")

NOTEBOOKS_DIR = notebook_candidates[0]

print("\nFINAL PATHS")
print("USW_DIR:", USW_DIR)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("NOTEBOOKS_DIR:", NOTEBOOKS_DIR)

print("\nCHECKS")
print("USW_DIR exists:", USW_DIR.exists())
print("PROJECT_ROOT exists:", PROJECT_ROOT.exists())
print("NOTEBOOKS_DIR exists:", NOTEBOOKS_DIR.exists())

In [ ]:
# 3. GitHub connection check and clone/pull public data-analysis branch

from pathlib import Path
import subprocess
import shutil

USE_GITHUB = True

GITHUB_USERNAME = "Al-Amin95"
GITHUB_REPOSITORY = "PromptInjectionDetectionSystem"
GITHUB_BRANCH = "data-analysis"
GITHUB_NOTEBOOK_PATH = "notebooks/1-data-analysis.ipynb"

GITHUB_REPO_URL = f"https://github.com/{GITHUB_USERNAME}/{GITHUB_REPOSITORY}"
GITHUB_CLONE_URL = f"{GITHUB_REPO_URL}.git"

REPO_LOCAL_PATH = Path("/content/PromptInjectionDetectionSystem")

print("GitHub connection information")
print("Repository:", GITHUB_REPO_URL)
print("Branch:", GITHUB_BRANCH)
print("Notebook path:", GITHUB_NOTEBOOK_PATH)
print("Repository visibility: Public")

def run_command(command, cwd=None):
    result = subprocess.run(
        command,
        cwd=cwd,
        capture_output=True,
        text=True
    )

    if result.stdout.strip():
        print(result.stdout.strip())

    if result.returncode != 0:
        if result.stderr.strip():
            print(result.stderr.strip())
        raise RuntimeError(f"Command failed: {' '.join(command)}")

if REPO_LOCAL_PATH.exists() and not (REPO_LOCAL_PATH / ".git").exists():
    print("\nFolder exists but is not a Git repository. Removing it.")
    shutil.rmtree(REPO_LOCAL_PATH)

if REPO_LOCAL_PATH.exists():
    print("\nRepository exists. Pulling latest changes.")
    run_command(["git", "remote", "set-url", "origin", GITHUB_CLONE_URL], cwd=REPO_LOCAL_PATH)
    run_command(["git", "fetch", "origin"], cwd=REPO_LOCAL_PATH)
    run_command(["git", "checkout", GITHUB_BRANCH], cwd=REPO_LOCAL_PATH)
    run_command(["git", "pull", "origin", GITHUB_BRANCH], cwd=REPO_LOCAL_PATH)
else:
    print("\nRepository not found. Cloning repository.")
    run_command(
        ["git", "clone", "-b", GITHUB_BRANCH, GITHUB_CLONE_URL, str(REPO_LOCAL_PATH)],
        cwd="/content"
    )

current_branch = subprocess.run(
    ["git", "branch", "--show-current"],
    cwd=REPO_LOCAL_PATH,
    capture_output=True,
    text=True,
    check=True
).stdout.strip()

print("\nGitHub repository ready.")
print("Repository local path:", REPO_LOCAL_PATH)
print("Repository exists:", REPO_LOCAL_PATH.exists())
print("Current Git branch:", current_branch)

if current_branch != GITHUB_BRANCH:
    raise ValueError(f"Wrong branch. Expected {GITHUB_BRANCH}, but got {current_branch}.")

In [ ]:
# 4-Verify GitHub branch and raw dataset files

RAW_DATA_DIR = REPO_LOCAL_PATH / "data" / "raw"

current_branch = subprocess.run(
    ["git", "branch", "--show-current"],
    cwd=REPO_LOCAL_PATH,
    capture_output=True,
    text=True,
    check=True
).stdout.strip()

raw_files = sorted([file for file in RAW_DATA_DIR.iterdir() if file.is_file()])
parquet_files = [file for file in raw_files if file.suffix == ".parquet"]

print("Repository local path:", REPO_LOCAL_PATH)
print("Repository exists:", REPO_LOCAL_PATH.exists())
print("Current Git branch:", current_branch)
print("Raw data directory:", RAW_DATA_DIR)
print("RAW_DATA_DIR exists:", RAW_DATA_DIR.exists())
print("Number of raw files:", len(raw_files))
print("Number of parquet dataset files:", len(parquet_files))

print("\nRaw dataset files:")
for file in raw_files:
    print(file.name)

if current_branch != GITHUB_BRANCH:
    raise ValueError(f"Wrong branch. Expected {GITHUB_BRANCH}, but got {current_branch}.")

if len(parquet_files) < 2:
    raise FileNotFoundError("Expected train and test parquet files were not found in data/raw.")

print("\nGitHub branch and raw dataset verification completed successfully.")

In [ ]:
# 5-Import libraries for data analysis

import os
import sys
import re
import json
import random
import platform
from pathlib import Path
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd

import matplotlib
import matplotlib.pyplot as plt

print("Libraries imported successfully.")

In [ ]:
# 6-Set random seed

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

print(f"Random seed set to: {SEED}")

In [ ]:
# 7-Record environment information

environment_info = {
    "python_version": sys.version,
    "platform": platform.platform(),
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "matplotlib_version": matplotlib.__version__,
    "notebook_run_datetime": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "random_seed": SEED
}

for key, value in environment_info.items():
    print(f"{key}: {value}")

In [ ]:
# 8-Define project paths


DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
DATA_ANALYSIS_OUTPUT_DIR = OUTPUTS_DIR / "data_analysis"

TABLES_DIR = DATA_ANALYSIS_OUTPUT_DIR / "tables"
FIGURES_DIR = DATA_ANALYSIS_OUTPUT_DIR / "figures"
REPORTS_DIR = DATA_ANALYSIS_OUTPUT_DIR / "reports"
LOGS_DIR = DATA_ANALYSIS_OUTPUT_DIR / "logs"

MODELS_DIR = PROJECT_ROOT / "models"
APP_DIR = PROJECT_ROOT / "webapp"

print("Project paths defined successfully.")
print("Project root:", PROJECT_ROOT)
print("Notebooks directory:", NOTEBOOKS_DIR)
print("Raw data directory:", RAW_DATA_DIR)
print("Processed data directory:", PROCESSED_DATA_DIR)
print("Tables directory:", TABLES_DIR)
print("Figures directory:", FIGURES_DIR)
print("Reports directory:", REPORTS_DIR)
print("Logs directory:", LOGS_DIR)
print("Models directory:", MODELS_DIR)
print("App directory:", APP_DIR)

In [ ]:
# 9-Create required folders

project_dirs = [
    PROJECT_ROOT,
    NOTEBOOKS_DIR,
    DATA_DIR,
    RAW_DATA_DIR,
    PROCESSED_DATA_DIR,
    OUTPUTS_DIR,
    DATA_ANALYSIS_OUTPUT_DIR,
    TABLES_DIR,
    FIGURES_DIR,
    REPORTS_DIR,
    LOGS_DIR,
    MODELS_DIR,
    APP_DIR
]

for directory in project_dirs:
    directory.mkdir(parents=True, exist_ok=True)

print("Required folders created/confirmed successfully.")

In [ ]:
# 10-Check folder structure

folder_check = []

for directory in project_dirs:
    folder_check.append({
        "folder_name": directory.name,
        "folder_path": str(directory),
        "exists": directory.exists()
    })

folder_check_df = pd.DataFrame(folder_check)

folder_check_df

In [ ]:
# 11-Save folder structure check

folder_check_path = LOGS_DIR / "folder_structure_check.csv"

folder_check_df.to_csv(folder_check_path, index=False)

print("Folder structure check saved to:")
print(folder_check_path)
print("File exists:", folder_check_path.exists())

In [ ]:
# 12-Save analysis_config.json

analysis_config = {
    "project_title": "Prompt Injection Detection Using Fine-Tuned Transformer Models: A Glass-Box Approach with Explainable AI",
    "notebook_name": "1-data-analysis.ipynb",
    "analysis_stage": "Part 1 - Data Analysis",
    "dataset_name": "deepset/prompt-injections",
    "task": "Binary prompt injection classification",
    "label_mapping": {
        "0": "SAFE",
        "1": "INJECTION"
    },
    "random_seed": SEED,
    "raw_data_policy": "Read-only in Part 1. No permanent modification in this notebook.",
    "github_repository": GITHUB_REPO_URL,
    "github_branch": GITHUB_BRANCH,
    "github_notebook_path": GITHUB_NOTEBOOK_PATH,
    "project_root": str(PROJECT_ROOT),
    "notebooks_dir": str(NOTEBOOKS_DIR),
    "raw_data_dir": str(RAW_DATA_DIR),
    "processed_data_dir": str(PROCESSED_DATA_DIR),
    "tables_dir": str(TABLES_DIR),
    "figures_dir": str(FIGURES_DIR),
    "reports_dir": str(REPORTS_DIR),
    "logs_dir": str(LOGS_DIR),
    "models_dir": str(MODELS_DIR),
    "app_dir": str(APP_DIR),
    "folder_structure_check_file": str(folder_check_path),
    "created_at": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "python_version": sys.version,
    "numpy_version": np.__version__,
    "pandas_version": pd.__version__,
    "matplotlib_version": matplotlib.__version__
}

config_path = LOGS_DIR / "analysis_config.json"

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(analysis_config, f, indent=4)

print("Analysis configuration saved to:")
print(config_path)
print("File exists:", config_path.exists())

# Part-2: Raw Dataset Verification and Loading analysis

In [ ]:
# 1-Set pandas display options

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 300)
pd.set_option("display.width", 120)

print("Pandas display options configured successfully.")

In [ ]:
# 2-Set matplotlib defaults

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["axes.grid"] = True

print("Matplotlib defaults configured successfully.")

In [ ]:
# 3-Check raw data directory exists
print("Raw data directory:", RAW_DATA_DIR)
print("RAW_DATA_DIR exists:", RAW_DATA_DIR.exists())

In [ ]:
# 3-List files inside raw data directory

raw_files = sorted([file for file in RAW_DATA_DIR.iterdir() if file.is_file()])

print("Number of files in raw data directory:", len(raw_files))
print("\nRaw data files:")

for file in raw_files:
    print(file.name)

In [ ]:
# 4-Define raw train and test file paths

RAW_TRAIN_PATH = RAW_DATA_DIR / "train-00000-of-00001-9564e8b05b4757ab.parquet"
RAW_TEST_PATH = RAW_DATA_DIR / "test-00000-of-00001-701d16158af87368.parquet"

print("Raw train path:", RAW_TRAIN_PATH)
print("Raw test path:", RAW_TEST_PATH)

In [ ]:
# 5-Check raw train and test files exist

print("Raw train file exists:", RAW_TRAIN_PATH.exists())
print("Raw test file exists:", RAW_TEST_PATH.exists())

In [ ]:
# 6-Load raw train dataset

train_df = pd.read_parquet(RAW_TRAIN_PATH)

print("Raw train dataset loaded successfully.")
print("Train dataset shape:", train_df.shape)

In [ ]:
# 7-Load raw test dataset

test_df = pd.read_parquet(RAW_TEST_PATH)

print("Raw test dataset loaded successfully.")
print("Test dataset shape:", test_df.shape)

In [ ]:
# 8-Check train and test dataset shapes

print("Train dataset shape:", train_df.shape)
print("Test dataset shape:", test_df.shape)

print("\nTrain rows:", train_df.shape[0])
print("Train columns:", train_df.shape[1])

print("\nTest rows:", test_df.shape[0])
print("Test columns:", test_df.shape[1])

In [ ]:
# 9- Display first few rows of train and test datasets

print("First 5 rows of train dataset:")
display(train_df.head())

print("\nFirst 5 rows of test dataset:")
display(test_df.head())

In [ ]:
# 10-Check column names

print("Train dataset columns:")
print(train_df.columns.tolist())

print("\nTest dataset columns:")
print(test_df.columns.tolist())

In [ ]:
# 11-Check data types

print("Train dataset data types:")
print(train_df.dtypes)

print("\nTest dataset data types:")
print(test_df.dtypes)

#Part-3: Combined EDA Dataset Creation Analysis

In [ ]:
# 1-2 Create temporary train and test dataset copies with split labels

train_eda_df = train_df.copy()
test_eda_df = test_df.copy()

train_eda_df["split"] = "train"
test_eda_df["split"] = "test"

print("Temporary train EDA dataset created.")
print("Temporary test EDA dataset created.")
print("Train EDA shape:", train_eda_df.shape)
print("Test EDA shape:", test_eda_df.shape)

In [ ]:
# 3-Combine train and test only for exploratory data analysis

combined_eda_df = pd.concat([train_eda_df, test_eda_df], ignore_index=True)

print("Combined EDA dataset created successfully.")
print("Combined EDA shape:", combined_eda_df.shape)

In [ ]:
# 4-Check combined dataset shape

print("Combined EDA dataset shape:", combined_eda_df.shape)

print("\nSplit counts:")
print(combined_eda_df["split"].value_counts())

print("\nCombined EDA columns:")
print(combined_eda_df.columns.tolist())

In [ ]:
# 5-Confirm combined dataset is for EDA only

print("Phase 3 read-only confirmation:")
print("combined_eda_df is a temporary dataset for exploratory data analysis only.")
print("combined_eda_df will not be used directly for fine-tuning.")
print("Raw train and test datasets have not been modified.")
print("Original train_df shape:", train_df.shape)
print("Original test_df shape:", test_df.shape)
print("Combined EDA shape:", combined_eda_df.shape)

#Part-4: Label Verification and Class Distribution Analysis

In [ ]:
# 1-Check unique labels in train, test, and combined data

train_unique_labels = sorted(train_df["label"].dropna().unique().tolist())
test_unique_labels = sorted(test_df["label"].dropna().unique().tolist())
combined_unique_labels = sorted(combined_eda_df["label"].dropna().unique().tolist())

print("Train unique labels:", train_unique_labels)
print("Test unique labels:", test_unique_labels)
print("Combined unique labels:", combined_unique_labels)

In [ ]:
# 2-Confirm labels are only 0 and 1

allowed_labels = {0, 1}

train_label_check = set(train_unique_labels).issubset(allowed_labels)
test_label_check = set(test_unique_labels).issubset(allowed_labels)
combined_label_check = set(combined_unique_labels).issubset(allowed_labels)

print("Train labels are valid:", train_label_check)
print("Test labels are valid:", test_label_check)
print("Combined labels are valid:", combined_label_check)

if not train_label_check:
    raise ValueError("Train dataset contains labels outside 0 and 1.")

if not test_label_check:
    raise ValueError("Test dataset contains labels outside 0 and 1.")

if not combined_label_check:
    raise ValueError("Combined EDA dataset contains labels outside 0 and 1.")

print("All labels are valid binary labels.")

In [ ]:
# 3-Define label mapping

label_mapping = {
    0: "SAFE",
    1: "INJECTION"
}

label_mapping_df = pd.DataFrame(
    list(label_mapping.items()),
    columns=["label", "label_name"]
)

print("Label mapping defined successfully.")
display(label_mapping_df)

In [ ]:
# 4-Count labels by split

label_count_df = (
    combined_eda_df
    .groupby(["split", "label"])
    .size()
    .reset_index(name="count")
)

label_count_df["label_name"] = label_count_df["label"].map(label_mapping)

print("Label counts by split:")
display(label_count_df)

In [ ]:
# 5-Calculate label percentages by split

label_distribution_df = label_count_df.copy()

label_distribution_df["total_in_split"] = label_distribution_df.groupby("split")["count"].transform("sum")
label_distribution_df["percentage"] = (
    label_distribution_df["count"] / label_distribution_df["total_in_split"] * 100
).round(2)

label_distribution_df = label_distribution_df[
    ["split", "label", "label_name", "count", "total_in_split", "percentage"]
]

print("Label distribution by split:")
display(label_distribution_df)

In [ ]:
# 6-Compare train and test class balance

class_balance_comparison_df = label_distribution_df.pivot(
    index=["label", "label_name"],
    columns="split",
    values="percentage"
).reset_index()

class_balance_comparison_df["percentage_difference_train_minus_test"] = (
    class_balance_comparison_df["train"] - class_balance_comparison_df["test"]
).round(2)

print("Train and test class balance comparison:")
display(class_balance_comparison_df)

In [ ]:
# 7-Save label mapping and label distribution tables

label_mapping_path = TABLES_DIR / "label_mapping.csv"
label_distribution_path = TABLES_DIR / "label_distribution_by_split.csv"
class_balance_path = TABLES_DIR / "class_balance_comparison.csv"

label_mapping_df.to_csv(label_mapping_path, index=False)
label_distribution_df.to_csv(label_distribution_path, index=False)
class_balance_comparison_df.to_csv(class_balance_path, index=False)

print("Label mapping saved to:", label_mapping_path)
print("File exists:", label_mapping_path.exists())

print("\nLabel distribution saved to:", label_distribution_path)
print("File exists:", label_distribution_path.exists())

print("\nClass balance comparison saved to:", class_balance_path)
print("File exists:", class_balance_path.exists())

# Part-5: Label-Based Sample Analysis

In [ ]:
# 1-Display representative SAFE examples

safe_examples_df = (
    combined_eda_df[combined_eda_df["label"] == 0]
    .sample(n=10, random_state=SEED)
    .copy()
)

safe_examples_df["label_name"] = safe_examples_df["label"].map(label_mapping)

safe_examples_df = safe_examples_df[
    ["split", "label", "label_name", "text"]
].reset_index(drop=True)

print("Representative SAFE examples:")
display(safe_examples_df)

In [ ]:
# 2-Display representative INJECTION examples

injection_examples_df = (
    combined_eda_df[combined_eda_df["label"] == 1]
    .sample(n=10, random_state=SEED)
    .copy()
)

injection_examples_df["label_name"] = injection_examples_df["label"].map(label_mapping)

injection_examples_df = injection_examples_df[
    ["split", "label", "label_name", "text"]
].reset_index(drop=True)

print("Representative INJECTION examples:")
display(injection_examples_df)

In [ ]:
# 3-Check whether SAFE examples look like normal user requests

print("Manual inspection checklist for SAFE examples:")
print("1. SAFE examples should look like normal user requests.")
print("2. They should not ask the model to ignore instructions.")
print("3. They should not ask to reveal system, developer, hidden, private, or confidential instructions.")
print("4. They should not contain jailbreak-style behaviour.")

display(safe_examples_df)

In [ ]:
# 4-Check whether INJECTION examples contain adversarial or instruction-hijacking behaviour

print("Manual inspection checklist for INJECTION examples:")
print("1. INJECTION examples may contain instruction override attempts.")
print("2. They may ask the model to ignore previous instructions.")
print("3. They may ask to reveal system prompts, hidden prompts, secrets, or confidential information.")
print("4. They may contain jailbreak-style or role-hijacking behaviour.")

display(injection_examples_df)

In [ ]:
# 5-Save representative examples by label

representative_examples_df = pd.concat(
    [safe_examples_df, injection_examples_df],
    ignore_index=True
)

representative_examples_path = TABLES_DIR / "representative_examples_by_label.csv"

representative_examples_df.to_csv(representative_examples_path, index=False)

print("Representative examples saved to:", representative_examples_path)
print("File exists:", representative_examples_path.exists())

display(representative_examples_df)

#Part-6: Missing, Empty, and Invalid Text Analysis

In [ ]:
# 1-Check missing values in text and label

train_missing_values = train_df[["text", "label"]].isnull().sum()
test_missing_values = test_df[["text", "label"]].isnull().sum()
combined_missing_values = combined_eda_df[["text", "label"]].isnull().sum()

print("Train missing values:")
print(train_missing_values)

print("\nTest missing values:")
print(test_missing_values)

print("\nCombined EDA missing values:")
print(combined_missing_values)

In [ ]:
# 2-Check empty text values

train_empty_text_count = train_df["text"].apply(lambda x: isinstance(x, str) and x == "").sum()
test_empty_text_count = test_df["text"].apply(lambda x: isinstance(x, str) and x == "").sum()
combined_empty_text_count = combined_eda_df["text"].apply(lambda x: isinstance(x, str) and x == "").sum()

print("Train empty text count:", train_empty_text_count)
print("Test empty text count:", test_empty_text_count)
print("Combined EDA empty text count:", combined_empty_text_count)

In [ ]:
# 3-Check whitespace-only text values

train_whitespace_text_count = train_df["text"].apply(lambda x: isinstance(x, str) and x.strip() == "").sum()
test_whitespace_text_count = test_df["text"].apply(lambda x: isinstance(x, str) and x.strip() == "").sum()
combined_whitespace_text_count = combined_eda_df["text"].apply(lambda x: isinstance(x, str) and x.strip() == "").sum()

print("Train whitespace-only text count:", train_whitespace_text_count)
print("Test whitespace-only text count:", test_whitespace_text_count)
print("Combined EDA whitespace-only text count:", combined_whitespace_text_count)

In [ ]:
# 4-Check non-string text values

train_non_string_text_count = train_df["text"].apply(lambda x: not isinstance(x, str)).sum()
test_non_string_text_count = test_df["text"].apply(lambda x: not isinstance(x, str)).sum()
combined_non_string_text_count = combined_eda_df["text"].apply(lambda x: not isinstance(x, str)).sum()

print("Train non-string text count:", train_non_string_text_count)
print("Test non-string text count:", test_non_string_text_count)
print("Combined EDA non-string text count:", combined_non_string_text_count)

In [ ]:
# 5-Check punctuation-only or symbol-only prompts

def is_punctuation_or_symbol_only(text):
    if not isinstance(text, str):
        return False

    stripped_text = text.strip()

    if stripped_text == "":
        return False

    return not any(character.isalnum() for character in stripped_text)

train_symbol_only_count = train_df["text"].apply(is_punctuation_or_symbol_only).sum()
test_symbol_only_count = test_df["text"].apply(is_punctuation_or_symbol_only).sum()
combined_symbol_only_count = combined_eda_df["text"].apply(is_punctuation_or_symbol_only).sum()

symbol_only_examples_df = combined_eda_df[
    combined_eda_df["text"].apply(is_punctuation_or_symbol_only)
][["split", "label", "text"]].copy()

print("Train punctuation/symbol-only text count:", train_symbol_only_count)
print("Test punctuation/symbol-only text count:", test_symbol_only_count)
print("Combined EDA punctuation/symbol-only text count:", combined_symbol_only_count)

print("\nPunctuation/symbol-only examples:")
display(symbol_only_examples_df)

In [ ]:
# 6-Check very short and long prompts

length_check_df = combined_eda_df[["split", "label", "text"]].copy()

length_check_df["character_length"] = length_check_df["text"].apply(lambda x: len(x) if isinstance(x, str) else 0)
length_check_df["word_count"] = length_check_df["text"].apply(lambda x: len(x.split()) if isinstance(x, str) else 0)

very_short_prompts_df = length_check_df[
    (length_check_df["character_length"] <= 10) | (length_check_df["word_count"] <= 2)
].copy()

longest_prompts_df = length_check_df.sort_values(
    by="character_length",
    ascending=False
).head(10).copy()

print("Very short prompt count:", len(very_short_prompts_df))
print("Longest prompt examples selected:", len(longest_prompts_df))

print("\nVery short prompt examples:")
display(very_short_prompts_df)

print("\nTop 10 longest prompt examples:")
display(longest_prompts_df)

In [ ]:
# 7-Save missing empty and invalid text analysis report

missing_invalid_report = {
    "train_missing_text": train_df["text"].isnull().sum(),
    "train_missing_label": train_df["label"].isnull().sum(),
    "test_missing_text": test_df["text"].isnull().sum(),
    "test_missing_label": test_df["label"].isnull().sum(),
    "combined_missing_text": combined_eda_df["text"].isnull().sum(),
    "combined_missing_label": combined_eda_df["label"].isnull().sum(),
    "train_empty_text_count": train_df["text"].apply(lambda x: isinstance(x, str) and x == "").sum(),
    "test_empty_text_count": test_df["text"].apply(lambda x: isinstance(x, str) and x == "").sum(),
    "combined_empty_text_count": combined_eda_df["text"].apply(lambda x: isinstance(x, str) and x == "").sum(),
    "train_whitespace_only_text_count": train_df["text"].apply(lambda x: isinstance(x, str) and x.strip() == "").sum(),
    "test_whitespace_only_text_count": test_df["text"].apply(lambda x: isinstance(x, str) and x.strip() == "").sum(),
    "combined_whitespace_only_text_count": combined_eda_df["text"].apply(lambda x: isinstance(x, str) and x.strip() == "").sum(),
    "combined_very_short_prompt_count": len(very_short_prompts_df),
    "longest_prompt_examples_selected": len(longest_prompts_df)
}

missing_invalid_report_df = pd.DataFrame(
    list(missing_invalid_report.items()),
    columns=["check", "count"]
)

missing_invalid_report_path = TABLES_DIR / "missing_empty_invalid_text_report.csv"
very_short_prompts_path = TABLES_DIR / "very_short_prompt_examples.csv"
longest_prompts_path = TABLES_DIR / "longest_prompt_examples.csv"

missing_invalid_report_df.to_csv(missing_invalid_report_path, index=False)
very_short_prompts_df.to_csv(very_short_prompts_path, index=False)
longest_prompts_df.to_csv(longest_prompts_path, index=False)

print("Missing/empty/invalid text report saved to:", missing_invalid_report_path)
print("File exists:", missing_invalid_report_path.exists())

print("\nVery short prompt examples saved to:", very_short_prompts_path)
print("File exists:", very_short_prompts_path.exists())

print("\nLongest prompt examples saved to:", longest_prompts_path)
print("File exists:", longest_prompts_path.exists())

display(missing_invalid_report_df)